# Snowflake Release Note Review Tracker

A lightweight recurring notebook to run monthly, per the JD's requirement to "monitor
Snowflake product releases and roadmap updates" and "review monthly release notes and assess
impacts to existing environments." This isn't a one-time audit like the others — it's meant
to be re-run and re-filled every review cycle.

### How this fits into the monthly cycle
1. Run Section 1 to snapshot current account context and parameters.
2. Read the current month's release notes (linked in Section 2).
3. For each item that could plausibly affect this environment, add a row to the log in
   Section 3.
4. For any **behavior change bundle** mentioned in the notes, use Section 4 to check its
   current status in this account before it becomes mandatory.
5. Carry open/in-progress rows forward into next month's copy of this notebook.


In [ ]:
-- ============================================================
-- 1. Account context snapshot — re-run each cycle to see what's changed since last review
-- ============================================================
SELECT
    CURRENT_ACCOUNT()          AS account,
    CURRENT_REGION()           AS region,
    CURRENT_VERSION()          AS snowflake_version,
    CURRENT_TIMESTAMP()        AS snapshot_taken_at;


In [ ]:
-- Full account parameter snapshot. Save/export this each cycle (or diff it against last
-- month's) to see exactly what configuration changed, which is often the fastest way to spot
-- a change that came from a release rather than from the team's own actions.
SHOW PARAMETERS IN ACCOUNT;


## 2. Where to check each cycle

- Snowflake Release Notes: `https://docs.snowflake.com/en/release-notes/new-features`
- Behavior Change Release Notes (separate from general release notes, and the higher-risk
  category since these change existing behavior rather than just adding new capability):
  `https://docs.snowflake.com/en/release-notes/behavior-changes`
- Snowsight itself also surfaces a "What's New" panel with account-relevant highlights.


## 3. Monthly review log

One row per feature/change assessed. Keep every cycle's rows in the same table (add a
`Review Month` column value) rather than starting a new table each month, so this becomes a
running history of what's been evaluated and decided.

| Review Month | Feature / Change | Category (GA / Preview / Behavior Change) | Affects Us? | Impact Area | Decision | Status | Owner | Target Date |
|---|---|---|---|---|---|---|---|---|
| *e.g. 2026-09* | *e.g. Example feature* | *GA* | *Yes/No* | *e.g. cost, security, ingestion* | *Adopt / Monitor / Not applicable* | *Not started / In progress / Done* | | |

**Decision definitions:**
- **Adopt** — worth implementing; create a follow-up task with a target date.
- **Monitor** — not urgent, but worth revisiting once it's more mature or once a dependent
  feature ships.
- **Not applicable** — reviewed and explicitly ruled out, so it doesn't get re-reviewed by
  accident next cycle.


In [ ]:
-- ============================================================
-- 4. Behavior change bundle status check
-- Replace 'YYYY_MM' with the bundle identifier named in the release notes you're reviewing.
-- Snowflake bundles behavior changes by release; this tells you whether a given bundle is
-- currently disabled, enabled, or in its default rollout state for this account.
-- ============================================================
SELECT SYSTEM$BEHAVIOR_CHANGE_BUNDLE_STATUS('YYYY_MM') AS bundle_status;


### Notes, caveats, and next steps

- **This notebook has no automated way to fetch the release notes themselves** — Snowsight
  notebooks can't reach external URLs, so the actual reading of `docs.snowflake.com` each
  month stays a manual step. Everything here is about capturing the *outcome* of that review
  in a structured, re-queryable way, and catching config drift via the parameter snapshot.
- **Keep last month's parameter snapshot** (export Section 1's second query, or copy it into
  a scratch table) so you can diff it against this month's — that turns "what changed" from a
  memory exercise into a query.
- **Behavior change bundles are the highest-priority category to track deliberately** — unlike
  new features, they change how existing queries/objects behave once a bundle becomes
  mandatory, which is exactly the kind of surprise a regulated environment can't afford.
